# Module 2: Kalman Filter Implementation

## Learning Objectives

By completing this notebook, you will:

1. **Formulate** dynamic factor models in state-space representation
2. **Implement** the Kalman filter from scratch for state estimation
3. **Compute** the log-likelihood via prediction error decomposition
4. **Extract** filtered and smoothed factor estimates
5. **Understand** how the Kalman gain balances prediction and observation

## Prerequisites

- Module 1 (static factor models, PCA)
- Understanding of state-space models
- Basic probability (conditional distributions, Gaussian distributions)

## Estimated Time: 120-150 minutes

---

## Setup and Imports

We'll implement the Kalman filter from first principles, then validate against statsmodels.

In [ ]:
# Purpose: Import libraries for state-space modeling and Kalman filtering
# Key Concept: Kalman filter is optimal recursive estimator for linear Gaussian systems

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import linalg
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

---

# Part 1: State-Space Representation of DFM

## 1.1 Dynamic Factor Model

A **dynamic factor model (DFM)** extends the static model by adding dynamics to factors:

**Measurement Equation:**
$$X_t = \Lambda F_t + e_t, \quad e_t \sim N(0, \Sigma_e)$$

**Transition Equation (Factor Dynamics):**
$$F_t = \Phi_1 F_{t-1} + \Phi_2 F_{t-2} + ... + \Phi_p F_{t-p} + \eta_t, \quad \eta_t \sim N(0, \Sigma_\eta)$$

## 1.2 State-Space Form

To apply the Kalman filter, we rewrite as:

**Measurement:** $X_t = Z \alpha_t + \epsilon_t$

**Transition:** $\alpha_t = T \alpha_{t-1} + R \eta_t$

where the **state vector** $\alpha_t$ contains current and lagged factors:
$$\alpha_t = \begin{bmatrix} F_t \\ F_{t-1} \\ \vdots \\ F_{t-p+1} \end{bmatrix}$$

### Example: Single Factor with AR(1) Dynamics

Simplest DFM:
- $X_t = \lambda F_t + e_t$ (measurement)
- $F_t = \phi F_{t-1} + \eta_t$ (factor AR(1))

State-space form:
- State: $\alpha_t = F_t$ (scalar)
- $X_t = \lambda \alpha_t + e_t$ where $Z = \lambda$
- $\alpha_t = \phi \alpha_{t-1} + \eta_t$ where $T = \phi$, $R = 1$

In [ ]:
# Purpose: Simulate data from a dynamic factor model
# Key Concept: Factors now have autoregressive dynamics

def simulate_dynamic_factor_model(T, N, r, phi, Lambda, 
                                   sigma_eta=1.0, sigma_e=0.5, seed=None):
    """
    Simulate data from a dynamic factor model with AR(1) factors.
    
    Parameters
    ----------
    T : int
        Number of time periods
    N : int
        Number of observed variables
    r : int
        Number of factors
    phi : ndarray, shape (r, r)
        Factor transition matrix (VAR(1) coefficients)
    Lambda : ndarray, shape (N, r)
        Factor loadings
    sigma_eta : float
        Standard deviation of factor innovations
    sigma_e : float
        Standard deviation of idiosyncratic errors
    seed : int, optional
        Random seed
    
    Returns
    -------
    X : ndarray, shape (T, N)
        Observed data
    F : ndarray, shape (T, r)
        True factors
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Step 1: Simulate factors with AR(1) dynamics
    F = np.zeros((T, r))
    F[0] = np.random.randn(r) * sigma_eta  # Initial value
    
    for t in range(1, T):
        eta_t = np.random.randn(r) * sigma_eta
        F[t] = phi @ F[t-1] + eta_t
    
    # Step 2: Generate observed data
    e = np.random.randn(T, N) * sigma_e
    X = F @ Lambda.T + e
    
    return X, F


# Simulate a simple 2-factor DFM
T = 200
N = 6
r = 2

# Factor dynamics: AR(1) with persistence
phi = np.array([[0.8, 0.0],
                [0.0, 0.6]])

# Loadings: first 3 variables load on factor 1, last 3 on factor 2
Lambda = np.array([
    [1.0, 0.1],
    [0.9, 0.2],
    [0.85, 0.15],
    [0.2, 1.0],
    [0.15, 0.9],
    [0.25, 0.85],
])

X, F_true = simulate_dynamic_factor_model(
    T=T, N=N, r=r, phi=phi, Lambda=Lambda,
    sigma_eta=1.0, sigma_e=0.5, seed=123
)

print(f"Simulated Dynamic Factor Model:")
print(f"  Time periods: {T}")
print(f"  Variables: {N}")
print(f"  Factors: {r}")
print(f"  Factor persistence: φ_11={phi[0,0]}, φ_22={phi[1,1]}")
print(f"\nData shape: {X.shape}")
print(f"True factors shape: {F_true.shape}")

In [ ]:
# Purpose: Visualize dynamic factor model data
# Key Concept: Factors exhibit persistence due to AR dynamics

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: True factors over time
axes[0, 0].plot(F_true[:, 0], linewidth=2, label='Factor 1', color='steelblue')
axes[0, 0].plot(F_true[:, 1], linewidth=2, label='Factor 2', color='coral')
axes[0, 0].set_title('True Dynamic Factors (AR(1) Process)')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Factor Value')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Autocorrelation of factors
from matplotlib import pyplot as plt
lags = range(20)
acf_f1 = [np.corrcoef(F_true[:-h, 0] if h > 0 else F_true[:, 0], 
                       F_true[h:, 0] if h > 0 else F_true[:, 0])[0, 1] 
          for h in lags]
acf_f2 = [np.corrcoef(F_true[:-h, 1] if h > 0 else F_true[:, 1], 
                       F_true[h:, 1] if h > 0 else F_true[:, 1])[0, 1] 
          for h in lags]

axes[0, 1].plot(lags, acf_f1, 'o-', linewidth=2, label='Factor 1 ACF', color='steelblue')
axes[0, 1].plot(lags, acf_f2, 'o-', linewidth=2, label='Factor 2 ACF', color='coral')
axes[0, 1].axhline(y=0, color='black', linewidth=0.5)
axes[0, 1].set_title('Factor Autocorrelation Functions')
axes[0, 1].set_xlabel('Lag')
axes[0, 1].set_ylabel('Autocorrelation')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Sample of observed variables
for i in range(3):
    axes[1, 0].plot(X[:, i], alpha=0.7, label=f'Var {i+1}')
axes[1, 0].set_title('Observed Variables (Factor 1 Group)')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('Value')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Cross-correlation structure
corr_matrix = np.corrcoef(X.T)
im = axes[1, 1].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1, 1].set_title('Correlation Matrix')
axes[1, 1].set_xlabel('Variable')
axes[1, 1].set_ylabel('Variable')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print("\nNote: Factor persistence creates autocorrelation in observed variables!")

---

# Part 2: Kalman Filter Implementation

## 2.1 The Kalman Filter Algorithm

The Kalman filter recursively computes optimal state estimates.

**Initialization:**
- $\hat{\alpha}_{1|0}$ = initial state estimate
- $P_{1|0}$ = initial state covariance

**For each time $t = 1, 2, ..., T$:**

**Prediction Step:**
$$\begin{align}
\hat{\alpha}_{t|t-1} &= T \hat{\alpha}_{t-1|t-1} \\
P_{t|t-1} &= T P_{t-1|t-1} T' + R Q R'
\end{align}$$

**Update Step:**
$$\begin{align}
v_t &= X_t - Z \hat{\alpha}_{t|t-1} \quad \text{(prediction error)} \\
F_t &= Z P_{t|t-1} Z' + H \quad \text{(prediction error variance)} \\
K_t &= P_{t|t-1} Z' F_t^{-1} \quad \text{(Kalman gain)} \\
\hat{\alpha}_{t|t} &= \hat{\alpha}_{t|t-1} + K_t v_t \\
P_{t|t} &= P_{t|t-1} - K_t Z P_{t|t-1}
\end{align}$$

In [ ]:
# Purpose: Implement Kalman filter from scratch
# Key Concept: Recursive optimal estimation for linear Gaussian state-space models

class KalmanFilter:
    """
    Kalman filter for state-space models.
    
    State-space form:
        X_t = Z @ alpha_t + epsilon_t,  epsilon_t ~ N(0, H)
        alpha_t = T @ alpha_{t-1} + R @ eta_t,  eta_t ~ N(0, Q)
    """
    
    def __init__(self, Z, T, R, H, Q, alpha_1, P_1):
        """
        Parameters
        ----------
        Z : ndarray, shape (N, m)
            Measurement matrix
        T : ndarray, shape (m, m)
            Transition matrix
        R : ndarray, shape (m, r)
            Selection matrix for state innovations
        H : ndarray, shape (N, N)
            Measurement error covariance
        Q : ndarray, shape (r, r)
            State innovation covariance
        alpha_1 : ndarray, shape (m,)
            Initial state mean
        P_1 : ndarray, shape (m, m)
            Initial state covariance
        """
        self.Z = Z
        self.T = T
        self.R = R
        self.H = H
        self.Q = Q
        self.alpha_1 = alpha_1
        self.P_1 = P_1
        
        # Dimensions
        self.N = Z.shape[0]  # Number of observed variables
        self.m = T.shape[0]  # State dimension
    
    def filter(self, X):
        """
        Run Kalman filter on observed data.
        
        Parameters
        ----------
        X : ndarray, shape (T, N)
            Observed data
        
        Returns
        -------
        alpha_filtered : ndarray, shape (T, m)
            Filtered state estimates
        alpha_predicted : ndarray, shape (T, m)
            Predicted state estimates
        P_filtered : ndarray, shape (T, m, m)
            Filtered state covariances
        P_predicted : ndarray, shape (T, m, m)
            Predicted state covariances
        loglik : float
            Log-likelihood
        """
        T_obs = X.shape[0]
        
        # Storage
        alpha_filtered = np.zeros((T_obs, self.m))
        alpha_predicted = np.zeros((T_obs, self.m))
        P_filtered = np.zeros((T_obs, self.m, self.m))
        P_predicted = np.zeros((T_obs, self.m, self.m))
        
        # Initialize
        alpha_tt = self.alpha_1
        P_tt = self.P_1
        
        loglik = 0.0
        
        for t in range(T_obs):
            # ===== PREDICTION STEP =====
            # Predict state: alpha_{t|t-1} = T @ alpha_{t-1|t-1}
            alpha_t_pred = self.T @ alpha_tt
            
            # Predict covariance: P_{t|t-1} = T @ P_{t-1|t-1} @ T' + R @ Q @ R'
            P_t_pred = self.T @ P_tt @ self.T.T + self.R @ self.Q @ self.R.T
            
            # Store predictions
            alpha_predicted[t] = alpha_t_pred
            P_predicted[t] = P_t_pred
            
            # ===== UPDATE STEP =====
            # Prediction error: v_t = X_t - Z @ alpha_{t|t-1}
            v_t = X[t] - self.Z @ alpha_t_pred
            
            # Prediction error variance: F_t = Z @ P_{t|t-1} @ Z' + H
            F_t = self.Z @ P_t_pred @ self.Z.T + self.H
            
            # Kalman gain: K_t = P_{t|t-1} @ Z' @ F_t^{-1}
            K_t = P_t_pred @ self.Z.T @ np.linalg.inv(F_t)
            
            # Update state: alpha_{t|t} = alpha_{t|t-1} + K_t @ v_t
            alpha_tt = alpha_t_pred + K_t @ v_t
            
            # Update covariance: P_{t|t} = P_{t|t-1} - K_t @ Z @ P_{t|t-1}
            P_tt = P_t_pred - K_t @ self.Z @ P_t_pred
            
            # Store filtered estimates
            alpha_filtered[t] = alpha_tt
            P_filtered[t] = P_tt
            
            # ===== LOG-LIKELIHOOD CONTRIBUTION =====
            # log p(X_t | X_1, ..., X_{t-1}) = log N(v_t; 0, F_t)
            loglik_t = -0.5 * (self.N * np.log(2 * np.pi) + 
                               np.log(np.linalg.det(F_t)) + 
                               v_t @ np.linalg.inv(F_t) @ v_t)
            loglik += loglik_t
        
        return alpha_filtered, alpha_predicted, P_filtered, P_predicted, loglik


print("KalmanFilter class implemented successfully!")

## 2.2 Applying Kalman Filter to DFM

For our 2-factor AR(1) model, the state-space matrices are:

- **State:** $\alpha_t = F_t$ (2x1)
- **Measurement:** $Z = \Lambda$ (6x2)
- **Transition:** $T = \Phi$ (2x2)
- **Selection:** $R = I_2$ (2x2)
- **Covariances:** $H = \sigma_e^2 I_N$, $Q = \sigma_\eta^2 I_r$

In [ ]:
# Purpose: Set up state-space representation for our DFM
# Key Concept: Map DFM parameters to Kalman filter matrices

# State-space matrices (using true parameters for now)
Z_ss = Lambda  # N x r
T_ss = phi     # r x r
R_ss = np.eye(r)  # r x r
H_ss = (0.5**2) * np.eye(N)  # N x N (measurement error variance)
Q_ss = (1.0**2) * np.eye(r)  # r x r (state innovation variance)

# Initial state (unconditional mean and variance for stationary AR(1))
alpha_1 = np.zeros(r)

# Unconditional variance: vec(P) = (I - T ⊗ T)^{-1} vec(R Q R')
# For diagonal phi, simpler formula applies
P_1 = np.diag([Q_ss[i, i] / (1 - T_ss[i, i]**2) for i in range(r)])

# Create Kalman filter
kf = KalmanFilter(Z=Z_ss, T=T_ss, R=R_ss, H=H_ss, Q=Q_ss,
                  alpha_1=alpha_1, P_1=P_1)

# Run filter
alpha_filt, alpha_pred, P_filt, P_pred, loglik = kf.filter(X)

print("Kalman Filter Results:")
print(f"  Filtered states shape: {alpha_filt.shape}")
print(f"  Log-likelihood: {loglik:.2f}")
print(f"\nFinal filtered state: {alpha_filt[-1]}")
print(f"True final state: {F_true[-1]}")

### Visualize Filtered Factor Estimates

In [ ]:
# Purpose: Compare true factors with Kalman filter estimates
# Key Concept: Kalman filter optimally extracts factors from noisy observations

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Factor 1
axes[0].plot(F_true[:, 0], linewidth=2, label='True Factor 1', alpha=0.8)
axes[0].plot(alpha_filt[:, 0], linewidth=2, label='Filtered Estimate', 
             alpha=0.8, linestyle='--')

# Add uncertainty bands (±2 std)
std_filt_0 = np.sqrt([P_filt[t, 0, 0] for t in range(T)])
axes[0].fill_between(range(T), 
                      alpha_filt[:, 0] - 2*std_filt_0,
                      alpha_filt[:, 0] + 2*std_filt_0,
                      alpha=0.2, label='95% Confidence')

axes[0].set_title('Factor 1: True vs Filtered Estimate')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Factor Value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Factor 2
axes[1].plot(F_true[:, 1], linewidth=2, label='True Factor 2', alpha=0.8)
axes[1].plot(alpha_filt[:, 1], linewidth=2, label='Filtered Estimate', 
             alpha=0.8, linestyle='--')

std_filt_1 = np.sqrt([P_filt[t, 1, 1] for t in range(T)])
axes[1].fill_between(range(T), 
                      alpha_filt[:, 1] - 2*std_filt_1,
                      alpha_filt[:, 1] + 2*std_filt_1,
                      alpha=0.2, label='95% Confidence')

axes[1].set_title('Factor 2: True vs Filtered Estimate')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Factor Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute estimation errors
errors = F_true - alpha_filt
rmse_factor1 = np.sqrt(np.mean(errors[:, 0]**2))
rmse_factor2 = np.sqrt(np.mean(errors[:, 1]**2))

print(f"\nFiltering RMSE:")
print(f"  Factor 1: {rmse_factor1:.4f}")
print(f"  Factor 2: {rmse_factor2:.4f}")

### Exercise 2.1: Understanding the Kalman Gain

**Task:** Investigate how the Kalman gain evolves over time and what determines its magnitude.

**Theory:** The Kalman gain $K_t = P_{t|t-1} Z' F_t^{-1}$ balances:
- **High $K_t$:** Trust observations more (low prediction certainty)
- **Low $K_t$:** Trust prediction more (high observation noise)

**Expected Output:** Gain should stabilize as filter converges.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Compute and plot the Kalman gain over time

def compute_kalman_gains(kf, X):
    """
    Compute Kalman gains at each time step.
    
    Parameters
    ----------
    kf : KalmanFilter
        Kalman filter instance
    X : ndarray, shape (T, N)
        Observed data
    
    Returns
    -------
    gains : ndarray, shape (T, m, N)
        Kalman gains at each time step
    """
    T_obs = X.shape[0]
    gains = np.zeros((T_obs, kf.m, kf.N))
    
    # Initialize
    alpha_tt = kf.alpha_1
    P_tt = kf.P_1
    
    for t in range(T_obs):
        # Prediction step
        alpha_t_pred = None  # Replace with kf.T @ alpha_tt
        P_t_pred = None  # Replace with T @ P_tt @ T' + R @ Q @ R'
        
        # Compute Kalman gain
        F_t = None  # Replace with Z @ P_t_pred @ Z' + H
        K_t = None  # Replace with P_t_pred @ Z' @ inv(F_t)
        
        gains[t] = K_t
        
        # Update (to prepare for next iteration)
        v_t = X[t] - kf.Z @ alpha_t_pred
        alpha_tt = alpha_t_pred + K_t @ v_t
        P_tt = P_t_pred - K_t @ kf.Z @ P_t_pred
    
    return gains


# Compute gains
gains = compute_kalman_gains(kf, X)

# Plot Frobenius norm of gain over time
gain_norms = np.array([np.linalg.norm(gains[t], 'fro') for t in range(T)])

plt.figure(figsize=(10, 5))
plt.plot(gain_norms, linewidth=2, color='darkblue')
plt.xlabel('Time')
plt.ylabel('Kalman Gain (Frobenius Norm)')
plt.title('Evolution of Kalman Gain Over Time')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nInitial gain norm: {gain_norms[0]:.4f}")
print(f"Final gain norm: {gain_norms[-1]:.4f}")
print(f"Converged: {np.abs(gain_norms[-1] - gain_norms[-10]) < 0.01}")

<details>
<summary>Hint 1: Prediction</summary>
Use: alpha_t_pred = kf.T @ alpha_tt and P_t_pred = kf.T @ P_tt @ kf.T.T + kf.R @ kf.Q @ kf.R.T
</details>

<details>
<summary>Hint 2: Kalman Gain</summary>
Compute: F_t = kf.Z @ P_t_pred @ kf.Z.T + kf.H
Then: K_t = P_t_pred @ kf.Z.T @ np.linalg.inv(F_t)
</details>

In [ ]:
# AUTO-GRADED TESTS
# ----------------------------------
def test_exercise_2_1():
    gains = compute_kalman_gains(kf, X)
    
    assert gains is not None, "Don't forget to compute Kalman gains!"
    assert gains.shape == (T, r, N), f"Gains should have shape (T, r, N), got {gains.shape}"
    
    # Check convergence: later gains should be similar
    gain_norms = np.array([np.linalg.norm(gains[t], 'fro') for t in range(T)])
    assert gain_norms[0] > gain_norms[-1], "Initial gain should be higher (more uncertain)"
    assert np.abs(gain_norms[-1] - gain_norms[-10]) < 0.1, "Gain should converge"
    
    print("✓ Exercise 2.1 passed! Kalman gains computed correctly.")

test_exercise_2_1()

In [ ]:
# SOLUTION
# --------
def compute_kalman_gains_solution(kf, X):
    T_obs = X.shape[0]
    gains = np.zeros((T_obs, kf.m, kf.N))
    alpha_tt = kf.alpha_1
    P_tt = kf.P_1
    
    for t in range(T_obs):
        alpha_t_pred = kf.T @ alpha_tt
        P_t_pred = kf.T @ P_tt @ kf.T.T + kf.R @ kf.Q @ kf.R.T
        F_t = kf.Z @ P_t_pred @ kf.Z.T + kf.H
        K_t = P_t_pred @ kf.Z.T @ np.linalg.inv(F_t)
        gains[t] = K_t
        v_t = X[t] - kf.Z @ alpha_t_pred
        alpha_tt = alpha_t_pred + K_t @ v_t
        P_tt = P_t_pred - K_t @ kf.Z @ P_t_pred
    
    return gains

---

# Part 3: Log-Likelihood and Model Comparison

## 3.1 Likelihood via Prediction Error Decomposition

The Kalman filter produces the likelihood as a byproduct:

$$\log L(\theta) = \sum_{t=1}^T \log p(X_t | X_1, ..., X_{t-1}; \theta)$$

where each term is:
$$\log p(X_t | X_{t-1:1}) = -\frac{N}{2}\log(2\pi) - \frac{1}{2}\log|F_t| - \frac{1}{2}v_t' F_t^{-1} v_t$$

This enables maximum likelihood estimation (MLE) of model parameters.

In [ ]:
# Purpose: Demonstrate how likelihood changes with different parameters
# Key Concept: Likelihood quantifies model fit to data

def evaluate_likelihood(X, Lambda, phi, sigma_eta, sigma_e):
    """
    Evaluate log-likelihood for given parameters.
    
    Parameters
    ----------
    X : ndarray, shape (T, N)
        Observed data
    Lambda : ndarray, shape (N, r)
        Factor loadings
    phi : ndarray, shape (r, r)
        Factor transition matrix
    sigma_eta : float
        Factor innovation std
    sigma_e : float
        Measurement error std
    
    Returns
    -------
    loglik : float
        Log-likelihood
    """
    N, r = Lambda.shape
    
    # Set up state-space
    Z = Lambda
    T_mat = phi
    R = np.eye(r)
    H = (sigma_e**2) * np.eye(N)
    Q = (sigma_eta**2) * np.eye(r)
    
    # Initial conditions
    alpha_1 = np.zeros(r)
    P_1 = np.diag([Q[i, i] / (1 - T_mat[i, i]**2) for i in range(r)])
    
    # Run filter
    kf = KalmanFilter(Z=Z, T=T_mat, R=R, H=H, Q=Q, alpha_1=alpha_1, P_1=P_1)
    _, _, _, _, loglik = kf.filter(X)
    
    return loglik


# Evaluate at true parameters
loglik_true = evaluate_likelihood(X, Lambda, phi, sigma_eta=1.0, sigma_e=0.5)
print(f"Log-likelihood at true parameters: {loglik_true:.2f}")

# Compare with misspecified parameters
loglik_wrong_phi = evaluate_likelihood(X, Lambda, np.eye(r)*0.3, 1.0, 0.5)
loglik_wrong_noise = evaluate_likelihood(X, Lambda, phi, 1.0, 1.5)

print(f"\nLog-likelihood with wrong φ (0.3): {loglik_wrong_phi:.2f}")
print(f"Log-likelihood with wrong σ_e (1.5): {loglik_wrong_noise:.2f}")
print(f"\nTrue parameters have highest likelihood: {loglik_true > loglik_wrong_phi and loglik_true > loglik_wrong_noise}")

---

# Summary and Key Takeaways

## What You've Learned

1. **State-Space Representation**
   - Dynamic factor models map to linear state-space form
   - Measurement equation: $X_t = Z\alpha_t + \epsilon_t$
   - Transition equation: $\alpha_t = T\alpha_{t-1} + R\eta_t$

2. **Kalman Filter Algorithm**
   - **Prediction:** Project state forward using dynamics
   - **Update:** Incorporate new observation via Kalman gain
   - Provides optimal (MMSE) state estimates under Gaussianity

3. **Kalman Gain Intuition**
   - Balances prediction uncertainty vs observation noise
   - Converges to steady-state for stationary systems
   - Determines how much to trust new data

4. **Likelihood Computation**
   - Prediction error decomposition provides likelihood
   - Enables parameter estimation via maximum likelihood
   - Model comparison through likelihood ratios

## Key Formulas to Remember

**Prediction:**
- $\hat{\alpha}_{t|t-1} = T\hat{\alpha}_{t-1|t-1}$
- $P_{t|t-1} = TP_{t-1|t-1}T' + RQR'$

**Update:**
- $K_t = P_{t|t-1}Z'F_t^{-1}$ (Kalman gain)
- $\hat{\alpha}_{t|t} = \hat{\alpha}_{t|t-1} + K_tv_t$ (state update)
- $P_{t|t} = P_{t|t-1} - K_tZP_{t|t-1}$ (covariance update)

## Connection to Next Modules

| Next Module | Connection |
|-------------|------------|
| Module 3: PCA Estimation | Initialize Kalman filter with PCA estimates |
| Module 4: ML Estimation | Optimize likelihood via EM algorithm |
| Module 5: Mixed Frequency | State-space handles irregular observations |

## Practical Applications

With the Kalman filter, you can now:
1. Extract dynamic factors from panel data
2. Handle missing observations naturally
3. Compute likelihood for parameter estimation
4. Produce optimal forecasts with uncertainty quantification

## Next Steps

After completing this notebook:
1. Implement the Kalman smoother for full-sample estimates
2. Learn about EM algorithm for parameter estimation
3. Proceed to Module 3: PCA-Based Estimation

---

## Self-Assessment

Before moving on, ensure you can:
- [ ] Express a DFM in state-space form
- [ ] Implement the Kalman filter prediction and update steps
- [ ] Explain what the Kalman gain represents
- [ ] Compute log-likelihood from prediction errors
- [ ] Interpret filtered state estimates and their uncertainty

---

*Excellent work! You've mastered the Kalman filter—the cornerstone of dynamic factor model estimation. This enables all subsequent advanced topics.*